# Quickstart with RAGStack

This notebook demonstrates how to set up a simple RAG pipeline with RAGStack. At the end of this notebook, you will have a fully functioning Question/Answer model that can answer questions using your supplied documents.

A RAG pipeline requires, at minimum, a vector store, an embedding model, and an LLM. In this tutorial, you will use an Astra DB vector store, an OpenAI embedding model, an OpenAI LLM, and LangChain to orchestrate it all together.

## Prerequisites

You will need a vector-enabled Astra database and an OpenAI Account.

* Create an [Astra vector database](https://docs.datastax.com/en/astra-serverless/docs/getting-started/create-db-choices.html).
* Create an [OpenAI account](https://openai.com/)
* Within your database, create an [Astra DB Access Token](https://docs.datastax.com/en/astra-serverless/docs/manage/org/manage-tokens.html) with Database Administrator permissions.
* Get your Astra DB Endpoint:
  * `https://<ASTRA_DB_ID>-<ASTRA_DB_REGION>.apps.astra.datastax.com`

See the [Prerequisites](https://docs.datastax.com/en/ragstack/docs/prerequisites.html) page for more details.

## Setup
`ragstack-ai` includes all the packages you need to build a RAG pipeline.

`datasets` is used to import a sample dataset

In [1]:
! pip install ragstack-ai datasets

  Using cached numpy-1.26.4.tar.gz (15.8 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'error'


  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [21 lines of output]
      + C:\Users\Piyush Agarwal\OneDrive\Desktop\chatbox\.venv\Scripts\python.exe C:\Users\Piyush Agarwal\AppData\Local\Temp\pip-install-rxb67eej\numpy_ad848226888744eebf14fac3bcbdb97c\vendored-meson\meson\meson.py setup C:\Users\Piyush Agarwal\AppData\Local\Temp\pip-install-rxb67eej\numpy_ad848226888744eebf14fac3bcbdb97c C:\Users\Piyush Agarwal\AppData\Local\Temp\pip-install-rxb67eej\numpy_ad848226888744eebf14fac3bcbdb97c\.mesonpy-vmf0am6j -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=C:\Users\Piyush Agarwal\AppData\Local\Temp\pip-install-rxb67eej\numpy_ad848226888744eebf14fac3bcbdb97c\.mesonpy-vmf0am6j\meson-python-native-file.ini
      The Meson build system
      Version: 1.2.99
      Source dir: C:\Users\Piyush Agarwal\AppData\Local\Temp\pip-install-rxb67eej\numpy_ad848226888744eebf14fac3bcbdb97c
      Build d

In [8]:
import os
from dotenv import load_dotenv
load_dotenv()

ASTRA_DB_APPLICATION_TOKEN = os.environ.get("ASTRA_DB_APPLICATION_TOKEN")
ASTRA_DB_API_ENDPOINT = os.environ.get("ASTRA_DB_API_ENDPOINT")                                                                 


## Create a vector store

You will create a vector store using Astra DB and an OpenAI embedding model.

In [9]:
from langchain_ollama import OllamaEmbeddings
from langchain_astradb import AstraDBVectorStore

embedding = OllamaEmbeddings(model="nomic-embed-text")
vstore = AstraDBVectorStore(
    embedding=embedding,
    collection_name="test",
    api_endpoint=ASTRA_DB_API_ENDPOINT,
    token=ASTRA_DB_APPLICATION_TOKEN,
)
print("Astra vector store configured")

Astra vector store configured


## Load a sample dataset

Load a sample dataset and store it in the vector store.

In [13]:
from datasets import load_dataset
from langchain_core.documents import Document

philo_dataset = load_dataset("datastax/philosopher-quotes")["train"]
print("An example entry:")
print(philo_dataset[16])

c:\Users\Piyush Agarwal\OneDrive\Desktop\chatbox\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Piyush Agarwal\.cache\huggingface\hub\datasets--datastax--philosopher-quotes. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 100%|██████████| 450/450 [00:00<00:00, 15251.28 exa

An example entry:
{'author': 'aristotle', 'quote': 'Love well, be loved and do something of value.', 'tags': 'love;ethics'}


In [17]:
docs = []
for entry in philo_dataset:
    metadata = {"author": entry["author"]}
    if entry["tags"]:
        metadata["tags"] = entry["tags"].split(";")  # ["knowledge", "ethics", "education"]
    doc = Document(page_content=entry["quote"], metadata=metadata)
    docs.append(doc)

print(f"Total documents to insert: {len(docs)}")

Total documents to insert: 450


In [18]:
inserted_ids = vstore.add_documents(docs)
print(f"\nInserted {len(inserted_ids)} documents.")


Inserted 450 documents.


## Run a similarity search

Test the vector store with a similarity search.

In [ ]:
results = vstore.similarity_search("Our life is what we make of it", k=3) #returns top 3most similar search
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

* The quality of life is determined by its activities. [{'author': 'aristotle'}]
* When we share - that is poetry in the prose of life. [{'author': 'freud', 'tags': ['love']}]
* Restlessness is the hallmark of existence. [{'author': 'schopenhauer'}]


## Build a RAG pipeline

Use LangChain to create a basic RAG pipeline with the vector store and an ollama.

In [24]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.llms import Ollama



prompt = ChatPromptTemplate.from_messages([
    ("system", """Answer the following question based on the context provided.
    Think step by step before providing the answer.
    <context>
    {context}
    </context>"""),
    ("user", "Question: {input}")
])

llm = Ollama(model="llama3.2")



In [25]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain  
retriever = vstore.as_retriever(search_kwargs={"k": 3})
document_chain = create_stuff_documents_chain(llm, prompt)
retrieval_chain = create_retrieval_chain(retriever, document_chain)

In [27]:
response = retrieval_chain.invoke({"input": "In the given context, what is the best view on how to deal with the hardships in life?"})
print(response["answer"])

Based on the provided context, the best view on how to deal with hardships in life is to "struggle." The context suggests that struggling (or finding meaning in suffering) is better than enduring or giving up. This implies that embracing challenges and finding ways to overcome them can lead to a more fulfilling and meaningful life, even if it's difficult.


## Cleanup

Optionally, delete the collection from Astra DB to clean up resources.

In [28]:
vstore.delete_collection()